# Imports

In [1]:
import ast
import yaml
import json
import requests
from collections import defaultdict
from lxml import etree
from elasticsearch import Elasticsearch
from bs4 import BeautifulSoup

# Constants

In [2]:
ES_HOST = "https://prj-ext-prod-erga-gcp-dr.es.europe-west2.gcp.elastic-cloud.com"
ES_USERNAME = "elastic"
ES_PASSWORD = "aCJuMoynlz190jYcw6Kb3gFE"

In [3]:
DATA_PORTAL_AGGREGATIONS = [
    "biosamples",
    "raw_data",
    "mapped_reads",
    "assemblies_status",
    "annotation_status",
    "annotation_complete",
    "project_name",
    "symbionts_assemblies_status",
    "symbionts_biosamples_status",
    "symbionts_raw_data_status",
]

In [4]:
es = Elasticsearch([ES_HOST], http_auth=("elastic", ES_PASSWORD))

In [24]:
es.indices.get_alias("*")

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_82218/1261066940.py:1: DeprecationWarning: Using positional arguments for APIs is deprecated and will be disabled in 8.0.0. Instead use only keyword arguments for all APIs. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.indices.get_alias("*")
/Users/alexey/.pyenv/versions/3.12.6/lib/python3.12/site-packages/elasticsearch/connection/base.py:200: ElasticsearchWarning: this request accesses system indices: [.security-tokens-7, .kibana_usage_counters_8.17.1_001, .kibana_ingest_8.13.1_001, .kibana_task_manager_8.13.1_001, .security-profile-8, .fleet-enrollment-api-keys-7, .apm-agent-configuration, .fleet-agents-7, .kibana_alerting_cases_8.13.1_001, .apm-custom-link, .tasks, .kibana_analytics_8.13.1_001, .fleet-policies-7, .fleet-servers-7, .security-7, .kibana_security_session_1, .kibana_security_solution_8.13.1_001, .fleet-policies-leader-7, .kibana_8.13.1_001], but in a future major version, 

{'2025-08-20_tracking_status': {'aliases': {}},
 '2025-10-18_data_portal': {'aliases': {}},
 '2025-10-19_data_portal': {'aliases': {}},
 '2025-08-21_specimens': {'aliases': {}},
 '2025-05-20_data_portal': {'aliases': {}},
 '2025-10-15_data_portal': {'aliases': {}},
 '2025-10-31_specimens': {'aliases': {}},
 '2025-10-16_data_portal': {'aliases': {}},
 '2025-10-16_specimens': {'aliases': {}},
 '.ent-search-actastic-crawler2_robots_txts': {'aliases': {}},
 '2025-10-17_data_portal': {'aliases': {}},
 '.ent-search-actastic-workplace_search_pre_content_sources_v3': {'aliases': {}},
 '.ent-search-actastic-reindex_jobs_v3': {'aliases': {}},
 '.ent-search-actastic-workplace_search_role_mappings_v8': {'aliases': {}},
 '2025-08-06_specimens': {'aliases': {}},
 '2025-06-17_specimens': {'aliases': {}},
 '.ent-search-actastic-connectors_jobs_v5': {'aliases': {}},
 '2025-05-29_data_portal': {'aliases': {}},
 '2025-11-15_specimens': {'aliases': {}},
 '2025-10-15_specimens': {'aliases': {}},
 '2025-11-

In [7]:
def update_summary_index():
    es = Elasticsearch([ES_HOST], http_auth=("elastic", ES_PASSWORD))
    body = dict()
    body["aggs"] = dict()
    for aggregation_field in DATA_PORTAL_AGGREGATIONS:
        body["aggs"][aggregation_field] = {
            "terms": {"field": aggregation_field, "size": 20}
        }
        body["aggs"]["taxonomies"] = {
            "nested": {"path": f"taxonomies.kingdom"},
            "aggs": {
                "kingdom": {"terms": {"field": f"taxonomies.kingdom.scientificName"}}
            },
        }
    results = es.search(index="2026-02-09_data_portal", body=body)
    names_mapping = {
        "biosamples": "BioSamples - Submitted",
        "raw_data": "Raw Data - Submitted",
        "assemblies_status": "Assemblies - Submitted",
        "annotation_complete": "Annotation Complete - Done",
    }
    summary = dict()
    for key, aggs in results["aggregations"].items():
        try:
            for bucket in aggs["buckets"]:
                if bucket["key"] == "Done":
                    if key in names_mapping:
                        summary.setdefault("status", {})
                        summary["status"][names_mapping[key]] = bucket["doc_count"]
                elif bucket["key"] != "Waiting" and "symbionts" not in key:
                    summary.setdefault("projects", {})
                    summary["projects"][bucket["key"]] = bucket["doc_count"]
                elif bucket["key"] != "Waiting" and "symbionts" in key:
                    summary.setdefault("status", {})
                    summary["status"][f"Symbionts {bucket['key']}"] = bucket[
                        "doc_count"
                    ]
        except KeyError:
            for bucket in aggs["kingdom"]["buckets"]:
                summary.setdefault("phylogeny", {})
                summary["phylogeny"][bucket["key"]] = bucket["doc_count"]
    es.index(index="summary_test", body=summary, id="summary")

In [8]:
update_summary_index()

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_85460/830043104.py:15: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  results = es.search(index="2026-02-09_data_portal", body=body)
/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_85460/830043104.py:42: DeprecationWarning: The 'body' parameter is deprecated for the 'index' API and will be removed in a future version. Instead use the 'document' parameter. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index(index="summary_test", body=summary, id="summary")


# Importing annotations

In [27]:
annotations = defaultdict(list)
count = 0
for filename in  ["darwin_tree_of_life", "erga_bge", "erga_pilot"]:
    with open(f'/Users/alexey/ebi_projects/projects.ensembl.org/_data/{filename}/species.yaml', 'r') as yaml_file:
        yaml_data = yaml.safe_load(yaml_file)
        for record in yaml_data:
            annotation = dict()
            annotation['species'] = record['species']
            annotation['accession'] = record['accession']
            count += 1
            print(f"{count}\r", end='', flush=True)
            acc_response = requests.get(f"https://www.ebi.ac.uk/ena/browser/api/xml/{annotation['accession']}")
            try:
                root = etree.fromstring(acc_response.content)
            except etree.XMLSyntaxError:
                print(f"XMLSyntaxError: {annotation['accession']}")
                continue
            try:
                tax_id = root.find("ASSEMBLY").find("TAXON").find("TAXON_ID").text
            except AttributeError:
                if annotation['accession'] == "GCF_902459465.1":
                    tax_id = "7604"
                elif annotation['accession'] == "GCF_902652985.1":
                    tax_id = "6579"
            annotation['tax_id'] = tax_id
            try:
                annotation['annotation'] = {'GTF': record['annotation_gtf'], "GFF3": record['annotation_gff3']}
            except KeyError:
                annotation['annotation'] = {'GTF': None, "GFF3": None}
            try:
                annotation['proteins'] = {'FASTA': record['proteins']}
            except KeyError:
                annotation['proteins'] = {'FASTA': None}
            try:
                annotation['transcripts'] = {'FASTA': record['transcripts']}
            except KeyError:
                annotation['transcripts'] = {'FASTA': None}
            try:
                annotation['softmasked_genome'] = {'FASTA': record['softmasked_genome']}
            except KeyError:
                annotation['softmasked_genome'] = {'FASTA': None}
            try:
                annotation['repeat_library'] = {'FASTA': record['repeat_library']}
            except KeyError:
                annotation['repeat_library'] = None
            annotation['other_data'] = {'ftp_dumps': record['ftp_dumps']}
            try:
                annotation['view_in_browser'] = record['beta_link']
            except KeyError:
                try:
                    annotation['view_in_browser'] = record['ensembl_link']
                except KeyError:
                    annotation['view_in_browser'] = None
            annotations[annotation["tax_id"]].append(annotation)
len(annotations)

1051

1039

In [28]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for tax_id, annotation in annotations.items():
    body = dict()
    body['annotations'] = annotation
    es.index(index="annotation", body=body, id=tax_id)

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_73432/3278161194.py:5: DeprecationWarning: The 'body' parameter is deprecated for the 'index' API and will be removed in a future version. Instead use the 'document' parameter. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index(index="annotation", body=body, id=tax_id)


# Importing ToL QC

In [23]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
tolqc_data = requests.get("https://tolqc.cog.sanger.ac.uk/data.json?search=&sort=name&order=asc").json()
tolqc_dict = defaultdict(list)
for record in tolqc_data:
    link = f"https://tolqc.cog.sanger.ac.uk/{record['group']}/{record['_name']}"
    tolqc_dict[record["taxon"]].append(link)
for tax_id, links in tolqc_dict.items():
    body = dict()
    body["tol_qc_links"] = links
    es.index(index="tol_qc", body=body, id=tax_id)

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_10443/3393767358.py:1: DeprecationWarning: The 'http_auth' parameter is deprecated. Use 'basic_auth' or 'bearer_auth' parameters instead
  es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))


# Importing Genome Notes

In [17]:
with open("/Users/alexey/ebi_projects/biodiversity-data-ingestion/airflow/dags/genome_notes.json", "r") as f:
    genome_notes = json.load(f)

In [18]:
headerWithJWT = dict()
headerWithJWT['f1000-authbearer'] = requests.get("https://wellcomeopenresearch.org/api/token", headers={'content-type': 'application/json'}, timeout=10000).json()['token']
headerWithJWT['content-type'] = 'application/json'
treeOfLifeArticlesResponse = requests.get("https://gateway.f1000.com/gateway/231?details=true", headers=headerWithJWT, timeout=10000).json()

articleVersionIds = list()
for article_id in treeOfLifeArticlesResponse["members"][0]["ids"]:
    article_version_response = requests.get(f'https://article.f1000.com/articles?id={article_id}&publishedOnly=true', headers=headerWithJWT, timeout=10000).json() 
    articleVersionIds.append(article_version_response[0]["versionIds"][0])

In [ ]:
headerWithJWT['f1000-authbearer'] = requests.get("https://wellcomeopenresearch.org/api/token", headers={'content-type': 'application/json'}, timeout=10000).json()['token']
treeOfLifeArticlesList = list()
problems = list()
for article_id in articleVersionIds:
    print(f"Progress: {articleVersionIds.index(article_id)/len(articleVersionIds)*100}%\r", end='', flush=True)
    try:
        article_response = requests.get(f'https://article.f1000.com/versions?id={article_id}&publishedOnly=true', headers=headerWithJWT, timeout=10000).json()
        treeOfLifeArticlesList.append(article_response)
    except:
        headerWithJWT['f1000-authbearer'] = requests.get("https://wellcomeopenresearch.org/api/token", headers={'content-type': 'application/json'}, timeout=10000).json()['token']
        try:
            article_response = requests.get(f'https://article.f1000.com/versions?id={article_id}&publishedOnly=true', headers=headerWithJWT, timeout=10000).json()
            treeOfLifeArticlesList.append(article_response)
        except:
            print(f"Issues with {article_id}")
            problems.append(article_id)
            continue
len(treeOfLifeArticlesList)

Progress: 5.79950289975145%%%%

In [14]:
def clean_study_id(study_id):
    if '?' in study_id:
        return study_id.split("?")[0]
    if '&' in study_id:
        return study_id.split("&")[0]
    return study_id

In [15]:
genome_notes = defaultdict(list)
visited_studies = list()
for index, article in enumerate(treeOfLifeArticlesList):
    print(f"{index}\r", end='', flush=True)
    URL = article[0]["htmlUrl"]
    html_response = requests.get(URL)
    html_text = html_response.text
    soup_archive = BeautifulSoup(html_text, 'html.parser')
    for link in soup_archive.find_all('a'):
        href = link.get("href")
        if href is not None and 'PRJ' in href:
            genome_note = dict()
            study_id_1 = clean_study_id(href.split(":")[-1])
            study_id_2 = clean_study_id(href.split("/")[-1])
            study_id_3 = clean_study_id(href.split("%3D")[-1])
            study_id_4 = clean_study_id(href.split("/")[-2])
            chosen_study_id = None
            if study_id_1.startswith("PRJ"):
                chosen_study_id = study_id_1
            elif study_id_2.startswith("PRJ"):
                chosen_study_id = study_id_2
            elif study_id_3.startswith("PRJ"):
                chosen_study_id = study_id_3
            elif study_id_4.startswith("PRJ"):
                chosen_study_id = study_id_4
            else:
                print(href)
            if chosen_study_id not in visited_studies:
                visited_studies.append(chosen_study_id)
            else:
                continue
            study_response = requests.get(f"https://www.ebi.ac.uk/ena/browser/api/xml/{chosen_study_id}")
            root = etree.fromstring(study_response.content)
            tax_id = None
            try:
                tax_id = root.find("PROJECT").find("UMBRELLA_PROJECT").find("ORGANISM").find("TAXON_ID").text
            except AttributeError:
                try:
                    tax_id = root.find("PROJECT").find("SUBMISSION_PROJECT").find("ORGANISM").find("TAXON_ID").text
                except AttributeError:
                    print(chosen_study_id)
                    continue
            if tax_id is not None:
                genome_note['tax_id'] = tax_id
                genome_note['study_id'] = chosen_study_id
                genome_note['url'] = article[0]['pdfUrl'].split("/pdf")[0]
                genome_note['citeURL'] = f"https://doi.org/{article[0]['doi']}"
                genome_note['title'] = article[0]['title']
                genome_note['abstract'] = article[0]['abstractText']
                genome_note['figureURI'] = '#'
                for img in soup_archive.find_all('img'):
                    src = img.get("src")
                    if 'figure1.gif' in src:
                        genome_note['figureURI'] = src
                genome_note['caption'] = None
                for caption in soup_archive.find_all('div', {'class': 'caption'}):
                    caption_text = caption.h3.text
                    if 'Figure 1' in caption_text:
                        genome_note['caption'] = caption_text
                genome_notes[tax_id].append(genome_note)

PRJEB3308
PRJEB47820
PRJEB40665
PRJEB27320
PRJEB43743
1206

In [16]:
len(genome_notes)

1189

In [28]:
type(list(genome_notes.keys())[0])

str

In [19]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for tax_id, genome_note in genome_notes.items():
    if tax_id == "1594315":
        body = dict()
        body["articles"] = [genome_note[0]]
        es.index(index="genome_note", body=body, id=tax_id)
    else:
        body = dict()
        body["articles"] = genome_note
        es.index(index="genome_note", body=body, id=tax_id)

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_15937/1088646110.py:10: DeprecationWarning: The 'body' parameter is deprecated for the 'index' API and will be removed in a future version. Instead use the 'document' parameter. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index(index="genome_note", body=body, id=tax_id)
/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_15937/1088646110.py:6: DeprecationWarning: The 'body' parameter is deprecated for the 'index' API and will be removed in a future version. Instead use the 'document' parameter. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index(index="genome_note", body=body, id=tax_id)


In [17]:
    with open("genome_notes.jsonl", "w") as file:
        for tax_id, genome_note in genome_notes.items():
            body = dict()
            if tax_id == "1594315":
                body["articles"] = [genome_note[0]]
            else:
                body["articles"] = genome_note
            body["tax_id"] = tax_id
            file.write(f"{json.dumps(body)}\n")

# NBN Atlas

In [7]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [8]:
summary = get_samples("images", es)

In [11]:
summary["NHMUK014561654"]

{'images': ['https://www.ebi.ac.uk/biostudies/files/S-BIAD588/09052024_dtol_reupload/NHMUK014561654_1.JPG',
  'https://www.ebi.ac.uk/biostudies/files/S-BIAD588/09052024_dtol_reupload/NHMUK014561654_2.JPG',
  'https://www.ebi.ac.uk/biostudies/files/S-BIAD588/09052024_dtol_reupload/NHMUK014561654_3.JPG',
  'https://www.ebi.ac.uk/biostudies/files/S-BIAD588/09052024_dtol_reupload/NHMUK014561654_4.JPG']}

In [8]:
summary['summary']['max_year'] = 2025

In [9]:
es.index("landing-page-summary", summary["summary"], id="summary")

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_50543/958564871.py:1: DeprecationWarning: Using positional arguments for APIs is deprecated and will be disabled in 8.0.0. Instead use only keyword arguments for all APIs. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index("landing-page-summary", summary["summary"], id="summary")


{'_index': 'landing-page-summary',
 '_id': 'summary',
 '_version': 2,
 'result': 'updated',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 1,
 '_primary_term': 1}

In [7]:
data_portal["878968"]["tolqc_links"]

['https://tolqc.cog.sanger.ac.uk/darwin/insects/Rutpela_maculata',
 'https://tolqc.cog.sanger.ac.uk/tol/insects/Rutpela_maculata']

In [8]:
import requests

In [9]:
response = requests.get("https://tolqc.cog.sanger.ac.uk/darwin/insects/Rutpela_maculata")

In [14]:
# total = 0
# broken_links = []
# for tax_id, record in data_portal.items():
#     total += 1
#     print(total)
#     try:
#         for link in record["tolqc_links"]:
#             response = requests.get(link)
#             if response.status_code != 200:
#                 broken_links.append(link)
#     except KeyError:
#         continue

In [16]:
broken_links[:10]

['https://tolqc.cog.sanger.ac.uk/tol/insects/Triaxomera_parasitella',
 'https://tolqc.cog.sanger.ac.uk/tol/insects/Rutpela_maculata',
 'https://tolqc.cog.sanger.ac.uk/tol/insects/Cerastis_rubricosa',
 'https://tolqc.cog.sanger.ac.uk/darwin/insects/Nymphalis_io',
 'https://tolqc.cog.sanger.ac.uk/tol/insects/Inachis_io',
 'https://tolqc.cog.sanger.ac.uk/tol/insects/Apotomis_betuletana',
 'https://tolqc.cog.sanger.ac.uk/tol/insects/Apamea_sordens',
 'https://tolqc.cog.sanger.ac.uk/tol/insects/Pyrausta_nigrata',
 'https://tolqc.cog.sanger.ac.uk/tol/dicots/Succisa_pratensis',
 'https://tolqc.cog.sanger.ac.uk/tol/insects/Eudasyphora_cyanicolor']

In [26]:
for link in broken_links:
    if 'darwin' in link:
        print(link)

https://tolqc.cog.sanger.ac.uk/darwin/insects/Nymphalis_io
https://tolqc.cog.sanger.ac.uk/darwin/fungi/Umbilicaria_deusta
https://tolqc.cog.sanger.ac.uk/darwin/protists/Diacronema_lutheri
https://tolqc.cog.sanger.ac.uk/darwin/fungi/Stereocaulon_spathuliferum
https://tolqc.cog.sanger.ac.uk/darwin/algae/Synura_sp_CCAP960_5
https://tolqc.cog.sanger.ac.uk/darwin/protists/Isochrysis_galbana
https://tolqc.cog.sanger.ac.uk/darwin/arthropods/Geophilus_truncorum
https://tolqc.cog.sanger.ac.uk/darwin/dicots/Euphrasia_micrantha_x_Euphrasia_scottica
https://tolqc.cog.sanger.ac.uk/darwin/dicots/Euphrasia_anglica_x_Euphrasia_nemorosa
https://tolqc.cog.sanger.ac.uk/darwin/monocots/Alisma_plantago_aquatica
https://tolqc.cog.sanger.ac.uk/darwin/fungi/Boletus_edulis
https://tolqc.cog.sanger.ac.uk/darwin/algae/Chaetoceros_sp_PLY_617
https://tolqc.cog.sanger.ac.uk/darwin/dicots/Euphrasia_marshallii
https://tolqc.cog.sanger.ac.uk/darwin/insects/Osmia_bicornis_bicornis
https://tolqc.cog.sanger.ac.uk/darwin/

In [23]:
for link in broken_links:
    if len(broken_links[0].split("/")[-1].split("_")) > 2:
        print(link)

In [17]:
tolqc_data = requests.get("https://tolqc.cog.sanger.ac.uk/data.json", timeout=60).json()

In [18]:
    tolqc_dict = defaultdict(list)
    for record in tolqc_data:
        link = (f"https://tolqc.cog.sanger.ac.uk/{record['group']}/"
                f"{record['_name']}")
        tolqc_dict[record["taxon"]].append(link)

In [24]:
tolqc_data[0]

{'data_types': {'ont': 0,
  'bionano': 0,
  'rna-seq': 0,
  'pacbio': 1,
  'htag': 0,
  '10x': 1,
  'iso-seq': 0,
  'hic': 1},
 'family': 'Dolomedidae',
 'genus': 'Dolomedes',
 'group': '25g/arthropods',
 '_name': 'Dolomedes_plantarius',
 'specimens': 'qDolPla1',
 'name': 'Dolomedes plantarius',
 'phylum': 'Arthropoda',
 'asm': {'accession': 'GCA_907164885.2',
  'stage': 'RELEASED',
  'name': 'reference'},
 'taxon': '257759',
 'order': 'Araneae',
 'common_name': ''}

In [5]:
import requests
biosamples_root_url = "https://www.ebi.ac.uk/biosamples/samples"
project_tag = "ERGA"

In [6]:
samples = {}
if project_tag in ["ASG", "DTOL", "ERGA"]:
    first_url = (
        f"{biosamples_root_url}?size=200&filter="
        f"attr%3Aproject%20name%3A{project_tag}"
    )
    samples_response = requests.get(first_url, timeout=3600).json()
    while "_embedded" in samples_response:
        for sample in samples_response["_embedded"]["samples"]:
            sample["project_name"] = project_tag
            samples[sample["accession"]] = sample
        if "next" in samples_response["_links"]:
            samples_response = requests.get(
                samples_response["_links"]["next"]["href"], timeout=3600
            ).json()
        else:
            samples_response = requests.get(
                samples_response["_links"]["last"]["href"], timeout=3600
            ).json()

In [25]:
"SAMEA116132837" in samples

False

In [7]:
names = set()
for biosample_id, sample in samples.items():
    names.add(sample["characteristics"]["organism"][0]["text"])

In [8]:
len(names)

1229

In [12]:
data_portal["569046"]["project_name"]

['DTOL']

In [13]:
dp_names = set()
for tax_id, record in data_portal.items():
    if "ERGA" in record["project_name"]:
        dp_names.add(record["organism"])

In [18]:
for organism_name in dp_names:
    if organism_name not in names:
        print(organism_name)

Pleurochaete squarrosa
Pulsatilla halleri
Tetrao urogallus
Chelidurella thaleri


In [19]:
for organism_name in names:
    if organism_name not in dp_names:
        print(organism_name)

Chelidura thaleri
Tortella squarrosa
Anemone halleri


In [246]:
data_portal = get_samples("data_portal_test", es)
for tax_id, record in data_portal.items():
    if 'nbnatlas' in record:
        body = dict()
        body['commonName'] = record['commonName']
        body['commonNameSource'] = record['commonNameSource']
        body['nbnatlas'] = record['nbnatlas']
        es.index("nbn_atlas", body, id=tax_id)

# Images

In [14]:
images = requests.get("https://ftp.ebi.ac.uk/biostudies/fire/S-BIAD/588/S-BIAD588/Files/09052024_dtol_reupload_file_list.json").json()

In [15]:
images_data = defaultdict(list)
for image in images:
    URL = f"https://www.ebi.ac.uk/biostudies/files/S-BIAD588/{image['path']}"
    for record in image['attributes']:
        if record['name'] == 'NHMUK Barcode':
            images_data[record['value']].append(URL)
len(images_data)

5873

In [23]:
for nhmuk_id, images in images_data.items():
    body = dict()
    body["images"] = images
    es.index("images", body, id=nhmuk_id)

# Importing Data Portal metadata

In [12]:
data = list()
with open("/Users/alexey/erga_results.txt", "r") as f:
    for line in f:
        line = line.rstrip()
        data.append(ast.literal_eval(line))
len(data)

7990

In [13]:
names = defaultdict(int)
for record in data:
    for name in record['project_name']:
        names[name] += 1
names

defaultdict(int,
            {'DTOL': 6627,
             'Project Psyche': 887,
             'ERGA': 1153,
             '25 genomes': 28,
             'ATLASea': 31,
             'ERGA Pilot': 30,
             'ERGA BGE': 111,
             'CBP': 13,
             'ERGA Community Genomes': 1,
             'ERGA Swiss node': 1,
             'ENDEMIXIT': 1})

In [14]:
# To remove duplicated records
for record in data:
    visited_ids = {}
    new_records = {}
    ranks = {
        "Submitted to BioSamples": 1,
        "Raw Data - Submitted": 2,
        "Assemblies - Submitted": 3
    }
    for sample in record["records"]:
        if sample["accession"] not in visited_ids:
            visited_ids[sample["accession"]] = ranks[sample["trackingSystem"]]
            new_records[sample["accession"]] = sample
        else:
            if ranks[sample["trackingSystem"]] > visited_ids[sample["accession"]]:
                visited_ids[sample["accession"]] = ranks[sample["trackingSystem"]]
                new_records[sample["accession"]] = sample
    record["records"] = list(new_records.values())

In [15]:
# To remove duplicated runs
for record in data:
    visited_ids = set()
    new_runs = list()
    for exp in record["experiment"]:
        if exp["run_accession"] not in visited_ids:
            visited_ids.add(exp["run_accession"])
            new_runs.append(exp)
    record["experiment"] = new_runs

In [16]:
# To remove duplicated assemblies
for record in data:
    visited_ids = set()
    new_assemblies = list()
    for assmbl in record["assemblies"]:
        if assmbl["accession"] not in visited_ids:
            visited_ids.add(assmbl["accession"])
            new_assemblies.append(assmbl)
    record["assemblies"] = new_assemblies

In [17]:
# To remove duplicated symbionts assemblies
for record in data:
    visited_ids = set()
    new_assemblies = list()
    for assmbl in record["symbionts_assemblies"]:
        if assmbl["accession"] not in visited_ids:
            visited_ids.add(assmbl["accession"])
            new_assemblies.append(assmbl)
    record["symbionts_assemblies"] = new_assemblies

In [18]:
# To remove duplicated metagenomes assemblies
for record in data:
    visited_ids = set()
    new_assemblies = list()
    for assmbl in record["metagenomes_assemblies"]:
        if assmbl["accession"] not in visited_ids:
            visited_ids.add(assmbl["accession"])
            new_assemblies.append(assmbl)
    record["metagenomes_assemblies"] = new_assemblies

In [19]:
es_data = list()
for record in data:
    es_data.append({"index": {"_index": "data_portal_test", "_id": record['tax_id']}})
    es_data.append(record)
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for i in range(0, len(es_data), 1000):
    print(f"Working on {i}: {i+1000}")
    _ = es.bulk(body=es_data[i:i+1000])

Working on 0: 1000
Working on 1000: 2000
Working on 2000: 3000
Working on 3000: 4000
Working on 4000: 5000
Working on 5000: 6000
Working on 6000: 7000
Working on 7000: 8000
Working on 8000: 9000
Working on 9000: 10000
Working on 10000: 11000
Working on 11000: 12000
Working on 12000: 13000
Working on 13000: 14000
Working on 14000: 15000
Working on 15000: 16000


In [84]:
es.delete("tracking_status_index_test", id="13068")

{'_index': 'tracking_status_index_test',
 '_id': '13068',
 '_version': 9,
 'result': 'deleted',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 63133,
 '_primary_term': 1}

In [85]:
es.delete("data_portal_test", id="13068")

{'_index': 'data_portal_test',
 '_id': '13068',
 '_version': 12,
 'result': 'deleted',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 183216,
 '_primary_term': 1}

In [80]:
ids = list()
for record in data:
    ids.append(record['tax_id'])
len(ids)

7950

In [21]:
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [22]:
data_portal = get_samples("data_portal", es)

NameError: name 'es' is not defined

In [45]:
data_portal_new = get_samples("2025-02-12_data_portal", es)

In [46]:
data_portal_old = get_samples("2025-02-10_data_portal", es)

In [47]:
for organism in data_portal_old:
    if organism not in data_portal_new:
        print(organism)

3349935
3349936
517479
3349939


In [43]:
data_portal['876063_3126489']['goat_info']

{'url': 'https://goat.genomehubs.org/records?record_id=3126489&result=taxon&taxonomy=ncbi#Ochlodes sylvanus',
 'attributes': [{'name': 'genome_size',
   'value': 555626250,
   'count': 6,
   'aggregation_method': 'median',
   'aggregation_source': 'ancestor'}]}

In [1]:
data_portal['876063_3126489'].keys()

NameError: name 'data_portal' is not defined

In [18]:
for tax_id, record in data_portal.items():
    if 'goat_info' in record and record['goat_info'] is not None:
        es.index("goat", record['goat_info'], id=tax_id)

In [83]:
for tax_id in data_portal:
    if int(tax_id) not in ids:
        print(tax_id)

876063_3126489
13068


# Importing Specimens

In [259]:
specimens = list()
with open("/Users/alexey/ebi_projects/data-ingestion-apache-beam/specimens.jsonl", "r") as f:
    for line in f:
        line = line.rstrip()
        data_record = ast.literal_eval(line)
        specimens.append({"index": {"_index": "organisms_test", "_id": data_record['accession']}})
        specimens.append(data_record)
len(specimens)

106099

In [301]:
# for specimen in specimens:
#     es.index("organisms_test", specimen, id=specimen["accession"])

# Updating Summary Index

In [34]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))

In [35]:
body = dict()
body["aggs"] = dict()
for aggregation_field in DATA_PORTAL_AGGREGATIONS:
    body["aggs"][aggregation_field] = {
        "terms": {"field": aggregation_field, "size": 20}
    }
    body["aggs"]["taxonomies"] = {
        "nested": {"path": f"taxonomies.kingdom"},
        "aggs": {"kingdom": {
            "terms": {
                "field": f"taxonomies.kingdom.scientificName"
            }
        }
                }
    }


In [38]:
results = es.search(index="data_portal", body=body)
names_mapping = {
        "biosamples": "BioSamples - Submitted",
        "raw_data": "Raw Data - Submitted",
        "assemblies_status": "Assemblies - Submitted",
        "annotation_complete": "Annotation Complete - Done"
}
summary = dict()
for key, aggs in results["aggregations"].items():
    try:
        for bucket in aggs["buckets"]:
            if bucket['key'] == 'Done':
                if key in names_mapping:
                    summary.setdefault("status", {})
                    summary["status"][names_mapping[key]] = bucket['doc_count']                
            elif bucket['key'] != 'Waiting' and "symbionts" not in key:
                summary.setdefault("projects", {})
                summary["projects"][bucket["key"]] = bucket['doc_count']
            elif bucket['key'] != 'Waiting' and "symbionts" in key:
                summary.setdefault("status", {})
                summary["status"][f"Symbionts {bucket['key']}"] = bucket['doc_count']
    except KeyError:
        for bucket in aggs["kingdom"]["buckets"]:
            summary.setdefault("phylogeny", {})
            summary["phylogeny"][bucket['key']] = bucket['doc_count']

In [39]:
es.index("summary_test", summary, id="summary")

{'_index': 'summary_test',
 '_id': 'summary',
 '_version': 22,
 'result': 'updated',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 21,
 '_primary_term': 2}

In [3]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [7]:
data_portal = get_samples("data_portal_test", es)

In [4]:
tracking_status = get_samples("tracking_status", es)

In [5]:
list(tracking_status.keys())[0]

'1448927'

In [37]:
from datetime import datetime, timedelta

In [33]:
datetime.now().strftime('%Y-%m-%d')

'2025-02-11'

In [35]:
datetime.today().strftime("%Y-%m-%d")

'2025-02-11'

In [38]:
yesterday_day_prefix = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")

In [39]:
yesterday_day_prefix

'2025-02-10'

In [11]:
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [31]:
data_portal = get_samples("data_portal", es)

In [32]:
count = 0
for tax_id, record in data_portal.items():
    if 'genome_notes' in record and len(record["genome_notes"]) > 0:
        count += 1
print(count)

1042


In [19]:
articles = list()
for tax_id, record in data_portal.items():
    print(f"{list(data_portal.keys()).index(tax_id)/len(data_portal)*100}\r", end='', flush=True)
    if 'genome_notes' in record and len(record["genome_notes"]) > 0:
        for article in record["genome_notes"]:
            article_response = requests.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/search?query={article['study_id']}&format=json").json()
            if len(article_response['resultList']['result']) > 0:
                pub_year = article_response['resultList']['result'][0]['pubYear']
                article['pub_year'] = pub_year
                article['pubYear'] = pub_year
            else:
                article['pub_year'] = None
                article['pubYear'] = None
            article['id'] = article['study_id']
            article['articleType'] = 'Genome Note'
            article['journalTitle'] = 'Wellcome Open Res'
            article['organism_name'] = record['organism']
            articles.append(article)

99.98756064187087574

In [16]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for i in range(0, len(articles), 10000):
    print(f"Working on {i}: {i+10000}")
    _ = es.bulk(body=articles[i:i+10000])

Working on 0: 10000


In [17]:
articles_old = get_samples("articles", es)

In [20]:
studies = list()
for art in articles:
    studies.append(art["study_id"])

In [21]:
for art_id in articles_old:
    if art_id not in studies:
        print(art_id)

In [30]:
len(set(studies))

1043

In [28]:
len(articles_old)

1043

In [32]:
data_portal = get_samples("data_portal", es)

In [55]:
annotations_ready = {}
for tax_id, record in data_portal.items():
    if record["currentStatus"] == "Annotation Complete":
        annotations_ready[tax_id] = record["organism"]

In [34]:
len(annotations_ready)

1045

In [52]:
data = list()
with open("/Users/alexey/Downloads/erga.jsonl", "r") as f:
    for line in f:
        line = line.rstrip()
        data.append(json.loads(line)["tax_id"])
len(data)

1039

In [57]:
for tax_id, organism in annotations_ready.items():
    if tax_id not in data:
        print(f"{tax_id}\t{organism}")

8839	Anas platyrhynchos
8869	Cygnus olor
2712997	Dendarus foraminosus
8954	Falco peregrinus
6063	Halichondria panicea
1000418	Galeopsis tetrahit
1100992	Eudonia mercurella
60567	Crossaster papposus
876063_3126489	Ochlodes sylvanus
219594	Aythya fuligula
1522478	Buglossoporus quercinus
1309588	Cydalima perspectalis
37040	Gavia stellata
53616	Lamellibrachia columna


In [60]:
data_portal["6063"].keys()

dict_keys(['tax_id', 'currentStatus', 'experiment', 'assemblies', 'project_name', 'records', 'taxonomies', 'organism', 'commonName', 'commonNameSource', 'symbionts_experiment', 'symbionts_assemblies', 'symbionts_analyses', 'symbionts_records', 'metagenomes_experiment', 'metagenomes_assemblies', 'metagenomes_records', 'annotation', 'biosamples', 'annotation_status', 'annotation_complete', 'assemblies_status', 'mapped_reads', 'raw_data', 'trackingSystem', 'tolid', 'orgGeoList', 'specGeoList', 'genome_notes', 'goat_info', 'nbnatlas', 'show_tolqc', 'tolqc_links'])

In [70]:
data_portal["53616"]["annotation"]

[{'species': 'Lamellibrachia columna',
  'accession': 'GCA_963662155.1',
  'tax_id': '53616',
  'annotation': {'GTF': 'https://ftp.ebi.ac.uk/pub/ensemblorganisms/Lamellibrachia_columna/GCA_963662155.1/ensembl/geneset/2024_01/genes.gtf.gz',
   'GFF3': 'https://ftp.ebi.ac.uk/pub/ensemblorganisms/Lamellibrachia_columna/GCA_963662155.1/ensembl/geneset/2024_01/genes.gff3'},
  'proteins': {'FASTA': 'https://ftp.ebi.ac.uk/pub/ensemblorganisms/Lamellibrachia_columna/GCA_963662155.1/ensembl/geneset/2024_01/pep.fa.gz'},
  'transcripts': {'FASTA': 'https://ftp.ebi.ac.uk/pub/ensemblorganisms/Lamellibrachia_columna/GCA_963662155.1/ensembl/geneset/2024_01/cdna.fa.gz'},
  'softmasked_genome': {'FASTA': 'https://ftp.ebi.ac.uk/pub/ensemblorganisms/Lamellibrachia_columna/GCA_963662155.1/genome/softmasked.fa.gz'},
  'repeat_library': None,
  'other_data': {'ftp_dumps': 'https://ftp.ebi.ac.uk/pub/ensemblorganisms/Lamellibrachia_columna/GCA_963662155.1/'},
  'view_in_browser': None}]

In [26]:
from lxml import etree

In [28]:
response = requests.get("https://www.ebi.ac.uk/ena/browser/api/xml/PRJEB61747")
root = etree.fromstring(response.content)

In [39]:
for project in root.find("PROJECT").find("RELATED_PROJECTS"):
    try:
        acc = project.find("CHILD_PROJECT").get("accession")
        response2 = requests.get(f"https://www.ebi.ac.uk/ena/browser/api/xml/{acc}")
        root2 = etree.fromstring(response2.content)
        tax_id = root2.find("PROJECT").find("UMBRELLA_PROJECT").find("ORGANISM").find("TAXON_ID").text
        if data_portal[tax_id]['currentStatus'] == "Submitted to BioSamples":
            print(f"{tax_id=}\t{acc=}")
    except AttributeError:
        continue

tax_id='378980'	acc='PRJEB79978'
tax_id='673926'	acc='PRJEB79981'
tax_id='3127771'	acc='PRJEB84162'
tax_id='3238344'	acc='PRJEB84177'
tax_id='502525'	acc='PRJEB84180'
tax_id='76314'	acc='PRJEB94586'
tax_id='84658'	acc='PRJEB94589'
tax_id='314080'	acc='PRJEB94592'
tax_id='1072197'	acc='PRJEB94595'
tax_id='237119'	acc='PRJEB94598'
tax_id='2995250'	acc='PRJEB94601'
tax_id='1265421'	acc='PRJEB94604'
tax_id='612121'	acc='PRJEB94607'
tax_id='166372'	acc='PRJEB94610'
tax_id='268458'	acc='PRJEB94613'
tax_id='997549'	acc='PRJEB94616'
tax_id='3230685'	acc='PRJEB94619'
tax_id='1872043'	acc='PRJEB94622'
tax_id='209733'	acc='PRJEB94625'
tax_id='287327'	acc='PRJEB94628'
tax_id='2819898'	acc='PRJEB94631'
tax_id='1767299'	acc='PRJEB94634'
tax_id='2033273'	acc='PRJEB94637'
tax_id='3086330'	acc='PRJEB94640'
tax_id='1734902'	acc='PRJEB94643'
tax_id='753226'	acc='PRJEB94646'
tax_id='2034322'	acc='PRJEB96084'
tax_id='87261'	acc='PRJEB96179'
tax_id='3350010'	acc='PRJEB96181'
tax_id='926007'	acc='PRJEB96183'

In [1]:
for i in range(0):
    print(i)

In [2]:
import math

In [3]:
math.ceil(9/99)

1